# Notebook 27 - Gate G6: recovery efficiency (amended design)

Pre-registered at commit `307a7dd`; amended **before any G6 result** (5 seeds matching NB26, Friedman blocked by seed as the primary test, split F1/guarded outcomes, recovery AUC secondary). Frozen 40% structures from the 17b/20b registries. No new selection. No test access.

**Stages**
1. bootstrap and imports
2. frozen configuration and the pre-registration amendment (`RUN_ARCHITECTURES` lives here)
3. data, teachers, frozen evaluation subsample
4. evaluation helpers and frozen-structure checks
5. recovery runs (resumable; only fully complete method x seed cells are skipped)
6. units to threshold, Friedman test blocked by seed, gate verdict
7. figures

Run stages in order. After an interruption, rerun stages 1-4 then stage 5; partially written cells are discarded and redone. G6b can only pass once both architectures are complete.

In [ ]:
# Stage 1 - bootstrap and imports
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, yaml
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare, kruskal

REPO = Path("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
assert REPO.exists(), f"repo not found: {REPO}"
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.saber.bridge_ciciot import load_bridge
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.surgery import prune_cnn1d_channels
from src.saber.metrics import full_model_audit, action_weighted_boundary_inversion_rate

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"
OUT = R / "27_recovery_efficiency"
OUT.mkdir(parents=True, exist_ok=True)
print("repo:", REPO)
print("device:", DEVICE)


In [ ]:
# Stage 2 - frozen configuration and pre-registration amendment
SABER_CFG = yaml.safe_load(open(REPO / "config/saber.yaml"))

METHODS = ["random", "magnitude", "taylor", "fisher", "saber_v2"]
SEEDS = [101, 211, 307, 401, 503]
E_MAX = {"shallow": 8, "deep": 6}
MIN_W = {"shallow": int(SABER_CFG["groups"]["minimum_remaining_per_layer"]), "deep": 8}
SUBSET_FRACTION = 0.10
FAM_PLATEAU_FRACTION = 0.95
B2A_GUARD = 0.05

RUN_ARCHITECTURES = ["shallow", "deep"]   # <-- set to ["shallow"] or ["deep"] for staged runs

AMENDMENT = {
    "gate": "G6_recovery_efficiency",
    "original_prereg_commit": "307a7dd",
    "amended_before_any_G6_result": True,
    "amendments": [
        {"change": "seeds 3 -> 5, using the NB26 seed set [101,211,307,401,503]",
         "reason": "NB26 measured recovery-seed variance exceeding selection-method variance "
                   "(deep AWBIR 47% seed vs 10% method); 3 unblocked seeds per method is "
                   "underpowered for that variance structure, and sharing NB26 seeds makes "
                   "the two experiments directly comparable."},
        {"change": "primary test Kruskal-Wallis -> Friedman blocked by seed (Kendall W reported); "
                   "Kruskal-Wallis retained as an unblocked secondary",
         "reason": "every method is recovered under the identical seed set, so the design is "
                   "matched-block; blocking removes the dominant seed variance instead of "
                   "letting it mask method effects."},
        {"change": "outcomes split: units to family-F1 threshold, and units to threshold AND "
                   "benign-false-alert guard; guard-binding frequency reported",
         "reason": "NB26 shows shallow minimal-recovery benign_to_attack near the 0.05 guard, so "
                   "one combined outcome would conflate slow recovery with guard violation."},
        {"change": "secondary continuous outcome added: mean family macro-F1 across units (AUC)",
         "reason": "units-to-threshold is coarse and censored; the AUC uses the whole curve."},
    ],
    "primary_outcome": "recovery units (passes over the frozen 10% subset) to tau",
    "tau": ("family macro-F1 >= %.2f x plateau, plateau = median across ALL methods and seeds "
            "of the final-unit family macro-F1" % FAM_PLATEAU_FRACTION),
    "censoring": "runs not reaching tau by E_MAX recorded as E_MAX+1 (censored)",
    "G6a_pass": "Friedman p<0.05 (blocked by seed) AND median spread >= 1 unit, in >=1 architecture",
    "G6b_pass": "saber_v2 strictly lowest median units in >=1 architecture, within 1 unit elsewhere",
    "no_test_access": True, "no_new_selection": True,
    "methods": METHODS, "seeds": SEEDS, "e_max": E_MAX,
}
(OUT / "G6_PREREGISTRATION_AMENDED.json").write_text(json.dumps(AMENDMENT, indent=2))
print(json.dumps(AMENDMENT, indent=2))


In [ ]:
# Stage 3 - data, teachers, frozen evaluation subsample
TRAIN_LOADER, VAL_LOADER, _TEST_UNUSED, SHALLOW_TEACHER, CLASS_NAMES = load_bridge()
taxonomy = ciciot2023_taxonomy(CLASS_NAMES)
robust_graph = pd.read_csv(R / "14_risk_graph/asvg_edges_robust.csv")
N_CLASSES = len(CLASS_NAMES)


class DeepCNN1D(nn.Module):
    def __init__(self, n_classes=34):
        super().__init__()
        def blk(i, o):
            return [nn.Conv1d(i, o, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(o)]
        self.conv = nn.Sequential(*blk(1, 64), *blk(64, 128), nn.MaxPool1d(2),
                                  *blk(128, 128), *blk(128, 256))
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(256, n_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


TEACHERS = {}
if "shallow" in RUN_ARCHITECTURES:
    TEACHERS["shallow"] = SHALLOW_TEACHER.to(DEVICE).eval()
if "deep" in RUN_ARCHITECTURES:
    _dt = DeepCNN1D(N_CLASSES)
    _dt.load_state_dict(torch.load(REPO / "models/ciciot2023/deepcnn1d_g5_seed0.pt",
                                   map_location="cpu", weights_only=False)["state_dict"])
    TEACHERS["deep"] = _dt.to(DEVICE).eval()

Xv, Yv = VAL_LOADER.dataset.tensors
VAL_Y_ALL = Yv.numpy()
_rng = np.random.default_rng(0)
_idx = np.concatenate([_rng.permutation(np.where(VAL_Y_ALL == c)[0])[:4000]
                       for c in range(N_CLASSES) if (VAL_Y_ALL == c).sum() > 0])
EX_X = Xv[_idx].to(DEVICE)
EX_Y = VAL_Y_ALL[_idx]
EXAMPLE_INPUT = Xv[:8].float().to(DEVICE)
print("teachers:", list(TEACHERS), "| evaluation subsample rows:", len(_idx))


In [ ]:
# Stage 4 - evaluation helpers and frozen structure lookup
def forward_logits(model):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(EX_X[i:i + 8192].to(DEVICE)).cpu()
                          for i in range(0, len(EX_X), 8192)]).numpy()


T_LOGITS = {a: forward_logits(m) for a, m in TEACHERS.items()}


def audit_of(model, architecture):
    logits = forward_logits(model)
    audit = full_model_audit(logits, EX_Y, taxonomy, DEFAULT_COST_PROFILES)
    awbir, _ = action_weighted_boundary_inversion_rate(
        T_LOGITS[architecture], logits, EX_Y, robust_graph)
    audit["awbir"] = float(awbir)
    return audit


_train_y = TRAIN_LOADER.dataset.tensors[1].numpy()
_counts = np.bincount(_train_y, minlength=N_CLASSES)
_w = np.zeros_like(_counts, dtype=np.float64)
_w[_counts > 0] = 1.0 / np.sqrt(_counts[_counts > 0])
_w[_counts > 0] /= _w[_counts > 0].mean()
CLASS_W = torch.tensor(_w, dtype=torch.float32, device=DEVICE)
N_TRAIN = len(TRAIN_LOADER.dataset)


def removed_path(architecture, method):
    if architecture == "shallow":
        return R / f"17b_calibrated_checkpoint_freeze/{method}_r40cal_removed_groups.csv"
    return R / f"20b_depth_checkpoint_freeze/{method}_minimal_r40_removed_groups.csv"


def raw_student(architecture, method):
    removed = pd.read_csv(removed_path(architecture, method))
    prune_map = {str(layer): sorted(grp["channel_index"].astype(int).tolist())
                 for layer, grp in removed.groupby("module_path")}
    student, _ = prune_cnn1d_channels(
        TEACHERS[architecture], prune_map, EXAMPLE_INPUT,
        minimum_remaining_per_layer=MIN_W[architecture])
    return student.to(DEVICE)


METRIC_KEYS = ["fine_macro_f1", "family_macro_f1", "awbir", "attack_to_benign_rate",
               "benign_to_attack_rate", "hsr_balanced_soc", "ece15"]

for arch in RUN_ARCHITECTURES:
    for method in METHODS:
        assert removed_path(arch, method).exists(), f"missing frozen structure: {arch}/{method}"
print("all frozen structures found")


In [ ]:
# Stage 5 - recovery runs (resumable per architecture/method/seed)
CURVE = OUT / "recovery_curves.csv"


def complete_cells(frame):
    """Cells that hold every unit 0..E_MAX. Partially written cells are NOT complete."""
    if frame.empty:
        return set()
    counts = frame.groupby(["architecture", "method", "seed"])["unit"].nunique()
    return {(a, m, s) for (a, m, s), n in counts.items() if n == E_MAX[a] + 1}


if CURVE.exists():
    existing = pd.read_csv(CURVE)
    done = complete_cells(existing)
    keys = existing.set_index(["architecture", "method", "seed"]).index
    existing = existing[[k in done for k in keys]]      # drop interrupted cells
    rows = existing.to_dict("records")
else:
    rows, done = [], set()
print("complete cells found:", len(done), "| usable rows:", len(rows))

for arch in RUN_ARCHITECTURES:
    for method in METHODS:
        for seed in SEEDS:
            if (arch, method, seed) in done:
                continue
            torch.manual_seed(seed)
            np.random.seed(seed)
            student = raw_student(arch, method)
            audit0 = audit_of(student, arch)
            cell_rows = [{"architecture": arch, "method": method, "seed": seed, "unit": 0,
                          **{k: float(audit0[k]) for k in METRIC_KEYS}}]

            gen = torch.Generator().manual_seed(seed)
            subset = torch.randperm(N_TRAIN, generator=gen)[: int(N_TRAIN * SUBSET_FRACTION)]
            loader = torch.utils.data.DataLoader(
                torch.utils.data.Subset(TRAIN_LOADER.dataset, subset.tolist()),
                batch_size=1024, shuffle=True,
                generator=torch.Generator().manual_seed(seed))
            optimiser = torch.optim.Adam(student.parameters(), lr=1e-3)
            criterion = nn.CrossEntropyLoss(weight=CLASS_W)

            for unit in range(1, E_MAX[arch] + 1):
                student.train()
                for xb, yb in loader:
                    optimiser.zero_grad()
                    criterion(student(xb.float().to(DEVICE)), yb.to(DEVICE)).backward()
                    optimiser.step()
                audit = audit_of(student, arch)
                cell_rows.append({"architecture": arch, "method": method, "seed": seed,
                                  "unit": unit,
                                  **{k: float(audit[k]) for k in METRIC_KEYS}})

            # write once per COMPLETED cell: a crash mid-cell leaves no partial record
            rows.extend(cell_rows)
            pd.DataFrame(rows).to_csv(CURVE, index=False)
            done.add((arch, method, seed))
            print(f"{arch} {method} seed{seed}: famF1 {audit0['family_macro_f1']:.3f} -> "
                  f"{audit['family_macro_f1']:.3f} | awbir {audit['awbir']:.3f} | "
                  f"b2a {audit['benign_to_attack_rate']:.4f}")

curves = pd.DataFrame(rows)
print("curve rows:", len(curves), "| complete cells:", len(done))


In [ ]:
# Stage 6 - units to threshold, blocked test, gate verdict
curves = pd.read_csv(OUT / "recovery_curves.csv")

# only architectures whose cells are all complete enter the analysis
_counts = curves.groupby(["architecture", "method", "seed"])["unit"].nunique()
_complete = {(a, m, s) for (a, m, s), n in _counts.items() if n == E_MAX[a] + 1}
ANALYSED = [a for a in sorted(curves["architecture"].unique())
            if len([c for c in _complete if c[0] == a]) == len(METHODS) * len(SEEDS)]
_skipped = [a for a in sorted(curves["architecture"].unique()) if a not in ANALYSED]
if _skipped:
    print("SKIPPED (incomplete, rerun stage 5 for these):", _skipped)
assert ANALYSED, "no architecture has a complete method x seed grid yet"

summary = []
for arch in ANALYSED:
    arch_rows = curves[curves["architecture"] == arch]
    finals = arch_rows[arch_rows["unit"] == E_MAX[arch]]["family_macro_f1"]
    tau = FAM_PLATEAU_FRACTION * float(np.median(finals))
    for method in METHODS:
        for seed in SEEDS:
            run = arch_rows[(arch_rows["method"] == method) &
                            (arch_rows["seed"] == seed)].sort_values("unit")
            if run.empty:
                continue
            post = run[run["unit"] > 0]
            hit_f1 = post[post["family_macro_f1"] >= tau]
            hit_guarded = post[(post["family_macro_f1"] >= tau) &
                               (post["benign_to_attack_rate"] <= B2A_GUARD)]
            units_f1 = int(hit_f1["unit"].iloc[0]) if len(hit_f1) else E_MAX[arch] + 1
            units_guarded = int(hit_guarded["unit"].iloc[0]) if len(hit_guarded) else E_MAX[arch] + 1
            summary.append({
                "architecture": arch, "method": method, "seed": seed, "tau": tau,
                "units_to_tau_f1": units_f1, "units_to_tau_guarded": units_guarded,
                "guard_delayed": bool(units_guarded > units_f1),
                "censored_f1": bool(len(hit_f1) == 0),
                "censored_guarded": bool(len(hit_guarded) == 0),
                "auc_family_f1": float(post["family_macro_f1"].mean()),
                "auc_awbir": float(post["awbir"].mean()),
                "final_family_f1": float(post["family_macro_f1"].iloc[-1])})

units = pd.DataFrame(summary)
units.to_csv(OUT / "units_to_tau.csv", index=False)


def kendalls_w(chi2, n_blocks, k):
    return float(chi2 / (n_blocks * (k - 1))) if n_blocks and k > 1 else float("nan")


per_arch = {}
for arch in sorted(units["architecture"].unique()):
    sub = units[units["architecture"] == arch]
    pivot = sub.pivot_table(index="seed", columns="method", values="units_to_tau_f1").dropna()
    blocks = [pivot[m].values for m in METHODS if m in pivot.columns]
    _flat = np.concatenate(blocks) if blocks else np.array([])
    degenerate = bool(_flat.size == 0 or np.all(_flat == _flat[0]))
    if degenerate:
        # every run reached tau at the same unit: no variation for either test to detect
        friedman_p, friedman_chi2, kw_p = float("nan"), float("nan"), float("nan")
    else:
        with np.errstate(invalid="ignore", divide="ignore"):
            try:
                stat = friedmanchisquare(*blocks)
                friedman_p, friedman_chi2 = float(stat.pvalue), float(stat.statistic)
            except Exception:
                friedman_p, friedman_chi2 = float("nan"), float("nan")
            try:
                kw_p = float(kruskal(*[sub[sub["method"] == m]["units_to_tau_f1"].values
                                       for m in METHODS]).pvalue)
            except Exception:
                kw_p = float("nan")
    med = sub.groupby("method")["units_to_tau_f1"].median()
    auc = sub.groupby("method")["auc_family_f1"].mean()
    per_arch[arch] = {
        "tau": float(sub["tau"].iloc[0]),
        "friedman_p": friedman_p, "friedman_chi2": friedman_chi2,
        "kendalls_w": kendalls_w(friedman_chi2, len(pivot), len(blocks)),
        "kruskal_p_unblocked": kw_p,
        "median_units": {m: float(med.get(m, np.nan)) for m in METHODS},
        "median_spread": float(med.max() - med.min()),
        "best_method_units": str(med.idxmin()),
        "mean_auc_family_f1": {m: float(auc.get(m, np.nan)) for m in METHODS},
        "best_method_auc": str(auc.idxmax()),
        "guard_delayed_runs": int(sub["guard_delayed"].sum()),
        "censored_runs": int(sub["censored_f1"].sum()),
        "degenerate_no_variation": degenerate,
        "G6a": bool(friedman_p == friedman_p and friedman_p < 0.05
                    and (med.max() - med.min()) >= 1.0)}

g6a = any(v["G6a"] for v in per_arch.values())
strict = [a for a, v in per_arch.items()
          if v["best_method_units"] == "saber_v2"
          and list(v["median_units"].values()).count(v["median_units"]["saber_v2"]) == 1]
within_one = all(v["median_units"]["saber_v2"] - min(v["median_units"].values()) <= 1.0
                 for v in per_arch.values())
both_arch = set(per_arch) >= {"shallow", "deep"}

gate = {"gate": "G6_recovery_efficiency",
        "G6a_passed": bool(g6a),
        "G6b_passed": bool(both_arch and len(strict) > 0 and within_one),
        "architectures_analysed": sorted(per_arch),
        "verdict_complete": bool(both_arch),
        "per_architecture": per_arch,
        "amendment": AMENDMENT}
(OUT / "G6_recovery_efficiency_gate.json").write_text(json.dumps(gate, indent=2))
print(json.dumps(gate, indent=2))
_tbl = units.groupby(["architecture", "method"])[
    ["units_to_tau_f1", "units_to_tau_guarded", "auc_family_f1"]].median()
try:
    display(_tbl)
except NameError:
    print(_tbl.to_string())


In [ ]:
# Stage 7 - figures
archs = sorted(per_arch)
assert archs, 'run stage 6 first'

fig, axes = plt.subplots(1, len(archs), figsize=(5.4 * len(archs), 3.4), squeeze=False)
for ax, arch in zip(axes[0], archs):
    arch_rows = curves[curves["architecture"] == arch]
    for method in METHODS:
        med = arch_rows[arch_rows["method"] == method].groupby("unit")["family_macro_f1"].median()
        ax.plot(med.index, med.values, marker="o", lw=1.3, label=method)
    ax.axhline(per_arch[arch]["tau"], color="0.4", ls="--", lw=0.9)
    ax.set_title(f"{arch}: median of {len(SEEDS)} recovery seeds")
    ax.set_xlabel("recovery units")
    ax.set_ylabel("family macro-F1")
    ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(OUT / "G6_recovery_curves.png", dpi=200)
plt.show()

fig, axes = plt.subplots(1, len(archs), figsize=(5.4 * len(archs), 3.2), squeeze=False)
for ax, arch in zip(axes[0], archs):
    sub = units[units["architecture"] == arch]
    ax.boxplot([sub[sub["method"] == m]["units_to_tau_f1"].values for m in METHODS])
    ax.set_xticks(range(1, len(METHODS) + 1))
    ax.set_xticklabels(METHODS)
    _p = per_arch[arch]["friedman_p"]
    _lbl = "n/a" if _p != _p else f"{_p:.3f}"
    ax.set_title(f"{arch}: units to tau (Friedman p={_lbl})")
    ax.set_ylabel("recovery units")
    ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(OUT / "G6_units_to_tau.png", dpi=200)
plt.show()
print("figures written ->", OUT)


In [ ]:
import pandas as pd
c = pd.read_csv("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/"
                "results/saber/27_recovery_efficiency/recovery_curves.csv")
s = c[(c.architecture == "shallow") & (c.seed == 307)]
print(s.pivot_table(index="unit", columns="method",
                    values=["family_macro_f1", "benign_to_attack_rate"]).round(3).to_string())

In [ ]:
import pandas as pd
c = pd.read_csv("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/"
                "results/saber/27_recovery_efficiency/recovery_curves.csv")
bad = c[(c.unit > 0) & (c.benign_to_attack_rate > 0.10)]
print("collapse events:", len(bad), "of", len(c[c.unit > 0]), "post-recovery evaluations\n")
print("by architecture, seed and unit (count of methods hit, max 5):")
print(bad.groupby(["architecture", "seed", "unit"])["method"].count().to_string())
print("\nby architecture and method:")
print(bad.groupby(["architecture", "method"]).size().to_string())
print("\nseeds with zero collapses:")
allcells = c[c.unit > 0].groupby(["architecture", "seed"]).size().index
hit = set(map(tuple, bad[["architecture", "seed"]].drop_duplicates().values))
print([k for k in allcells if tuple(k) not in hit])

In [ ]:
%%bash
cd /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression
git add notebooks/27_recovery_efficiency.ipynb results/saber/27_recovery_efficiency
git commit -m "NB27 G6 FAILED: selection does not change recovery cost (all methods reach tau in 1 unit, both architectures; median spread 0; Friedman p=0.478 deep, 0.887 shallow; Kendall W 0.18/0.06). Any cost difference lies below the minimum measurable unit; no finer-grained rerun. UNPLANNED FINDING, seed-synchronous recovery collapse: 25 of 350 post-recovery evaluations show benign_to_attack>0.10, and 4 of 7 collapse events hit ALL FIVE methods at the IDENTICAL unit (deep s307 u4, deep s401 u2, shallow s307 u4 and u8); 2 of 5 seeds produce collapses on both architectures, 3 produce none; events are transient (next epoch recovers) and evenly distributed across methods (2-4 each). Shared seed implies shared subset and batch order, so collapse is a data-order effect independent of selection - this mechanism explains the NB26 recovery-seed variance and the NB20 vs 20b ranking swing. Deployment implication: fixed-epoch recovery can terminate on a collapsed epoch; per-epoch semantic monitoring and safety-gated checkpoint selection required"
git push origin saber-ids-method
git log --oneline -1

In [ ]:
%%bash
cd /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression
cat ../.gitconfig > /root/.gitconfig
cat ../.git-credentials > /root/.git-credentials && chmod 600 /root/.git-credentials
echo "gitconfig bytes: $(wc -c < /root/.gitconfig) | credentials bytes: $(wc -c < /root/.git-credentials)"
git config user.name; git config user.email
git branch --show-current
git log --oneline -1
git status --short | head -5